# Tech Challenge: Classificando a qualidade de vinhos com Machine Learning

Este notebook atende ao desafio de prever a qualidade de
um vinho com base em suas características físico-químicas.
- Quality >= 7: Vinho de alta qualidade.
- Quality < 7: Vinho de baixa/média qualidade.

## 1. Compreensão do Problema

- Variável alvo original: quality
- Transformação binária: quality_binary = 1 se quality >= 7, senão 0
- Objetivo: Treinar e avaliar modelos de aprendizado de máquina capazes de prever essa classificação a partir das variáveis disponíveis.

### Ferramentas e imports principais

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from matplotlib.colors import LinearSegmentedColormap

# Silenciador de avisos de atualizações futuras
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    )

In [ ]:
# Configura estilo dos gráficos para melhorar legibilidade.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Localiza a raiz do projeto tanto ao abrir pela raiz quanto pela pasta notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / 'data' / 'winequality-red.csv').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'winequality-red.csv'
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    raise FileNotFoundError(f'Dataset não encontrado em: {data_path}')

# Disponibiliza os módulos reutilizáveis de src/ ao executar pela raiz ou por notebooks/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.avaliacao_metricas import avaliar_modelo
from src.feature_engineering import adicionar_features

### Leitura do dataset de vinhos

In [ ]:
# Leitura do dataset de vinhos.
df = pd.read_csv(data_path)

# Transforma a qualidade original em alvo binário:
# 1 = alta qualidade (quality >= 7), 0 = baixa/media (quality < 7).
df['quality_binary'] = (df['quality'] >= 7).astype(int)

df.head()

In [ ]:
#Números representando  a dimensionalidade do conjunto de dados
df.shape

## 2. Análise Exploratória de Dados (EDA)



In [ ]:
# Visão geral da base.
print('Dimensão da base (linhas, colunas):', df.shape)
print('\nQuantidade de valores nulos por coluna:')
print(df.isnull().sum())

# Gráfico 1: balanceamento das classes.
# Ele mostra quantos vinhos há em cada classe do alvo binário.
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='quality_binary', hue='quality_binary', palette='Set2', legend=False)
ax.set_title('Balanceamento das Classes', fontsize=13, weight='bold')
ax.set_xlabel('Classe alvo (0 = baixa/media, 1 = alta)')
ax.set_ylabel('Quantidade de amostras')

# Adiciona os valores no topo de cada barra para leitura imediata.
for p in ax.patches:
    altura = int(p.get_height())
    ax.annotate(
        f'{altura}',
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center',
        va='bottom',
        fontsize=10,
        xytext=(0, 3),
        textcoords='offset points'
    )

plt.tight_layout()
plt.savefig(results_dir / 'balanceamento_classes.png', dpi=300, bbox_inches='tight')
plt.show()

proporcao = df['quality_binary'].value_counts(normalize=True).sort_index()
print('\nInterpretação do gráfico:')
print(f"- Classe 0 (baixa/media): {proporcao.loc[0]*100:.1f}%")
print(f"- Classe 1 (alta): {proporcao.loc[1]*100:.1f}%")
print('- Quanto mais desbalanceadas as barras, maior o cuidado na avaliação dos modelos.')

In [ ]:
# Gráfico 2: mapa de calor de correlações entre variáveis numéricas.
# Cores próximas de vermelho = correlação positiva forte.
# Cores próximas de azul = correlação negativa forte.

# Excluímos 'Id' e 'quality' para evitar redundâncias automáticas, pouco informativas e vazamento de dados (data leakage)
df_filtrado = df.drop(columns=['Id', 'quality'], errors='ignore')
corr = df_filtrado.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
ax = sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    linewidths=0.3,
    cbar_kws={'label': 'Coeficiente de correlação'}
)
ax.set_title('Matriz de Correlação das Variáveis (Filtrada)', fontsize=13, weight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(results_dir / 'matriz_correlacao.png', dpi=300, bbox_inches='tight')
plt.show()

# Mostra automaticamente as correlações mais fortes com o alvo para facilitar interpretação.
corr_alvo = corr['quality_binary'].drop('quality_binary').sort_values(key=np.abs, ascending=False)
print('Interpretação do gráfico (top 5 correlações reais com qualidade_binary):')
for nome, valor in corr_alvo.head(5).items():
    direcao = 'positiva' if valor >= 0 else 'negativa'
    print(f'- {nome}: correlação {direcao} ({valor:.3f})')

print('\nLeitura rápida: valores mais próximos de +1 ou -1 indicam relação mais forte com a classe alvo.')

### Distribuição das variáveis (histogramas)

In [ ]:
# Garantindo que a lista de variáveis esteja na memória da célula
variaveis_preditoras = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar","chlorides", "free sulfur dioxide", 
    "total sulfur dioxide","density", "pH", "sulphates", "alcohol"
]

# Garantindo que o 'X' seja recriado do df
X = df[variaveis_preditoras].copy() 

# Configurando os Histogramas
plt.figure(figsize=(16, 12)) # Tamanho da figura ajustado para melhor visualização dos histogramas
plt.suptitle("Distribuição das Variáveis Numéricas (Histogramas)", # Título do gráfico principal
              fontsize=18, 
              weight='bold', 
              color='#4a1525') 

for i, col in enumerate(variaveis_preditoras, 1): # Enumerate permite iterar sobre a lista de variáveis, fornecendo um índice (i) e o nome da coluna (col)
    plt.subplot(3, 4, i)
    sns.histplot(data=X, x=col, kde=True, color="#c7520f") 
    plt.title(f'Distribuição: {col}', fontsize=12, weight='semibold')
    plt.xlabel('')
    plt.ylabel('')

plt.tight_layout() # Ajusta o layout para evitar sobreposição de elementos
plt.savefig(results_dir / 'distribuicao_histogramas.png', dpi=300, bbox_inches='tight') # Salva o gráfico como imagem
plt.show() # Plota os histogramas

### Insights da análise exploratória (EDA)

Após a análise visual das distribuições e dispersões das características químicas dos vinhos, foram consolidados os seguintes apontamentos:

* **Assimetria nas distribuições (Histogramas):** Variáveis como `residual sugar` (açúcar residual) e `chlorides` (cloretos) apresentam uma distribuição altamente concentrada à esquerda com caudas longas à direita. Isso indica que a grande maioria dos vinhos da base possui características secas e baixa salinidade, com poucos lotes fora do padrão.
* **Presença crítica de outliers (boxplots):** Praticamente todas as variáveis numéricas apresentam pontos isolados além das cercas do boxplot, com destaque extremo para `residual sugar` e `chlorides`. Esses valores discrepantes são candidatos naturais para tratamento ou uso de modelos baseados em árvores, que são menos sensíveis a essas distorções.
* **Proximidade com a normalidade:** A variável `pH` e a variável `density` (densidade) são as únicas que se aproximam visualmente de uma curva Gaussiana perfeita (distribuição normal), indicando um controle padrão rígido de acidez global no processo de fabricação desses vinhos.

### Identificação de outliers (boxplots)

In [ ]:
# 2. Boxplots para Identificação de Outliers
plt.figure(figsize=(16, 12))
plt.suptitle("Análise de Outliers (Boxplots)", fontsize=18, weight='bold', color="#6b1730")

# Criando o grid de boxplots para as variáveis do modelo
for i, col in enumerate(variaveis_preditoras, 1):
    plt.subplot(3, 4, i)
    sns.boxplot(data=X, y=col, color="#74070d")
    plt.title(f'Outliers: {col}', fontsize=12, weight='semibold')
    plt.ylabel('')

plt.tight_layout()
plt.savefig(results_dir / 'analise_outliers_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Preparação dos Dados para Machine Learning

Nesta etapa, a base será preparada para o treinamento dos modelos de classificação. 
A variável `quality_binary` será utilizada como alvo, onde:

- 1 representa vinho de alta qualidade (`quality >= 7`);
- 0 representa vinho de baixa/média qualidade (`quality < 7`).

A coluna `quality` será removida das variáveis de entrada para evitar vazamento de informação, pois ela foi usada para criar a variável alvo binária.

In [ ]:
# Visualizando o DataFrame atual
display(df.head())

In [ ]:
# Conferência das colunas
print("Colunas da base:")
print(df.columns.tolist())

### Feature engineering

In [ ]:
# Cria as variáveis derivadas sem alterar o DataFrame original.
df_add = adicionar_features(df)

In [ ]:
# Visualizando df com novas variávariaveis
display(df_add.head())

In [ ]:
# Coletando as variáveis preditoras (independentes): características físicas-químicas do vinho.
variaveis_preditoras = [
    "fixed acidity","volatile acidity","citric acid","residual sugar",
    "chlorides","free sulfur dioxide","total sulfur dioxide",
    "density","pH","sulphates", "alcohol", "alcohol_density_ratio","total_acidity","volatile_fixed_ratio",
    "free_total_sulfur_ratio","sulphates_chlorides_ratio","alcohol_sulphates","alcohol_volatile_acidity"
]

In [ ]:
# Variável X contém as variáveis preditoras/independentes
X = df_add[variaveis_preditoras].copy()

# Variável que os modelos tentarão prever:
# Y é o target (dependente): 1 = alta qualidade e 0 = baixa/média qualidade.
y = df_add["quality_binary"].copy()

print("\nColunas usadas no modelo:")
print(X.columns.tolist())

print("\nDistribuição da variável alvo:")
print(y.value_counts())

print("\nProporção da variável alvo:")
print(y.value_counts(normalize=True))

In [ ]:
print("Quantidade de variáveis usadas no modelo:", len(X.columns))

for coluna in X.columns:
    print("-", coluna)

## 4. Separação entre Treino e Teste

A base será dividida entre treino e teste. 
Como existe desbalanceamento entre as classes, será utilizado o parâmetro `stratify=Y` para preservar a proporção de vinhos de alta qualidade e baixa/média qualidade nas duas amostras.

| Objeto    | Descrição                                     |
| --------- | -------------------------------------------------- |
| `X_train` | Dados usados para o modelo aprender                |
| `Y_train` | Respostas usadas no treinamento                    |
| `X_test`  | Dados usados para testar o modelo                  |
| `Y_test`  | Respostas verdadeiras para comparar com a previsão |


In [ ]:
# Dividindo os dados entre treino e teste
# Separando os dados entre treino e teste 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

## 5. Treinamento dos Modelos

Nesta etapa, serão treinados quatro modelos de classificação para comparar qual deles consegue prever melhor se um vinho é de **alta qualidade** ou de **baixa/média qualidade**.

Os modelos escolhidos foram:

1. **Regressão Logística**  
   Esse modelo foi escolhido por ser uma opção mais simples e fácil de entender.  
   Ele trabalha com a ideia de probabilidade, ou seja, tenta calcular a chance de um vinho pertencer a uma das classes, como por exemplo: qual a probabilidade desse vinho ser de alta qualidade?

2. **Árvore de Decisão**  
   Esse modelo ajuda a criar regras de decisão com base nas características dos vinhos.  
   Ele funciona como uma sequência de perguntas, por exemplo: o teor alcoólico é maior que determinado valor? A acidez está abaixo de certo limite?  
   Isso ajuda bastante na interpretação, porque conseguimos entender melhor quais condições influenciam a classificação.

3. **Random Forest**  
   O Random Forest é parecido com a Árvore de Decisão, mas em vez de usar apenas uma árvore, ele usa várias árvores ao mesmo tempo.  
   Cada árvore faz uma previsão, e no final o modelo combina essas respostas para chegar a uma classificação final.  
   Por isso, ele pode ser mais estável e menos dependente de uma única regra.

4. **SVM com RBF**  
   Esse modelo foi incluído para testar uma forma diferente de separar as classes.  
   Enquanto alguns modelos trabalham de forma mais direta, o SVM com RBF tenta encontrar uma separação mais flexível entre os vinhos de alta qualidade e os demais.  
   Isso pode ajudar quando a diferença entre as classes depende da combinação de várias características ao mesmo tempo.

A ideia de usar esses quatro modelos é comparar abordagens diferentes: uma simples, uma baseada em regras, uma que combina várias árvores e outra que tenta separar as classes de forma mais flexível.

## Modelo: Regressão logistica

In [ ]:
# Definindo a estrutura do pipeline, ajustnado hiperparâmetros básicos da Regressão logistica
lr_pipeline = Pipeline([                            # Tupla que recebe lista de tarefas a serem executadas sequencialmente
    ('imputer', SimpleImputer(strategy='median')),  # Substitui valores ausentes pela mediana da coluna
    ('scaler', StandardScaler()),                   # Padroniza os dados para média 0 e desvio padrão 1
    ('model', LogisticRegression(class_weight='balanced', # Ajusta o peso das classes para lidar com desbalanceamento
                                 C=1.0,             # Controla a regularização como "freio" (quanto maior, menos regularização)
                                 random_state=42))  # Semente aleatória para os cálculos se repetirem, garantindo reprodutibilidade dos resultados
])

### Treino

In [ ]:
# Treinando a nossa esteira completa com os dados de treino
# Método fit ajusta/treina o modelo com os dados de treino, aplicando as transformações definidas no pipeline.
# X_train é a tabela com as variáveis preditoras de treino (tipo caderno de questões)
# y_train é a variável alvo de treino (o gabarito com as respostas)
lr_pipeline.fit(X_train, y_train)

### Gerando as previsões 

In [ ]:
# Gerando palpites finais (0 ou 1) para o conjunto de teste
y_pred = lr_pipeline.predict(X_test) # Este método trás o resultado final

# Gerando as probabilidades de o vinho ser de alta qualidade (Classe 1)
y_proba = lr_pipeline.predict_proba(X_test)[:, 1] # Este método devolve o nível de certeza do modelo trazendo a segunda coluna (1)
                                                  # que é a métrica ROC pra avaliar a probabilidade da classe positiva (alta qualidade). 
                                                  # A primeira coluna (0) é a probabilidade da classe negativa (baixa/média qualidade).

### Avaliação de Métricas

In [ ]:
# Calcula, exibe e guarda as métricas da Regressão Logística.
metricas_lr = avaliar_modelo(y_test, y_pred, y_proba)

### Matriz de confusão

In [ ]:
# Matriz de confusão e o relatório resumido
print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

### Plotando as informações do modelo

In [ ]:
# Configurando o tema visual e o tamanho do painel
sns.set_theme(style="whitegrid") # Configura o estilo visual dos gráficos
plt.rcParams['figure.figsize'] = (15, 6) # Define o tamanho padrão dos gráficos para 15 polegadas de largura e 6 polegadas de altura

# Criando o esqueleto com 3 sub-gráficos lado a lado
fig, (ax1, ax2, ax3) = plt.subplots(1, 3) # Cria uma figura com 3 subplots lado a lado

# Adicionando título principal do modelo
plt.suptitle("Modelo de Regressão Logística", fontsize=20, weight='bold', color='#2b0a0a') # Adiciona título principal do modelo com fonte maior, negrito e cor personalizada

# Plotando a Matriz de Confusão no primeiro eixo
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax1, cmap='YlOrRd') # Plota a matriz de confusão usando as previsões do modelo e o conjunto de teste
ax1.set_title("Matriz de Confusão") # Titulo do gráfico da matriz de confusão

# Plotando a Curva ROC no segundo eixo
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax2, color='darkred') # Plota a curva ROC usando as probabilidades previstas pelo modelo e o conjunto de teste
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_title("Curva ROC (Sensibilidade)") # Titulo do gráfico da curva ROC

# Extraindo e plotando os coeficientes (impacto) no terceiro eixo
coefs = lr_pipeline.named_steps['model'].coef_[0] # Acessa os coeficientes do modelo de regressão logística treinado, que indicam a importância de cada variável preditora na decisão do modelo
features = X_train.columns                        # Acessa os nomes das colunas do conjunto de treino, que correspondem às variáveis preditoras usadas no modelo
importancia = pd.Series(coefs, index=features).sort_values() # Cria uma série pandas com os coeficientes do modelo, indexada pelos nomes das variáveis preditoras ordenadas
importancia.plot(kind='barh', ax=ax3, color='gold') # Plota um gráfico de barras horizontal mostrando a importância de cada característica na decisão do modelo
ax3.set_title("Impacto das Características") # Titulo do gráfico do impacto das características

# Ajustando o espaçamento entre os gráficos
plt.tight_layout()

# Salvando o gráfico automaticamente na pasta de resultados
plt.savefig(results_dir / 'regressao_logistica_outputs.png', dpi=300, bbox_inches='tight')

# Plotando o gráfico
plt.show()

### Análise de Desempenho: Regressão Logística

Após a inclusão das novas variáveis criadas no processo de *feature engineering*, a Regressão Logística apresentou acurácia de **79%** no conjunto de teste.

Para a classe **Alta**, o modelo obteve **recall de 66%**, indicando que conseguiu identificar boa parte dos vinhos realmente classificados como de alta qualidade. No entanto, a **precision foi de 36%**, mostrando que o modelo também classificou muitos vinhos de baixa/média qualidade como se fossem de alta qualidade.

A matriz de confusão mostra que o modelo acertou **21 dos 32 vinhos de alta qualidade**, mas também gerou **38 falsos positivos**, ou seja, vinhos de baixa/média qualidade classificados incorretamente como alta qualidade.

Dessa forma, a Regressão Logística mostrou boa capacidade de encontrar vinhos de alta qualidade, mas com menor precisão nas classificações positivas. Isso indica que o modelo tende a ser mais sensível, porém menos seletivo.

## Modelo: Árvore de decisão

In [ ]:
# Montando a esteira Pipeline e ajustando os hiperparâmetros da árvore
dt_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # Substitui valores ausentes pela mediana da coluna
    ('model', DecisionTreeClassifier(class_weight='balanced', # Ajusta o peso das classes para lidar com desbalanceamento
                                     max_depth=5,  # Limita a profundidade máxima da árvore para evitar overfitting (memorização excessiva dos dados de treino)
                                     random_state=42)) # Semente aleatória para os cálculos se repetirem, garantindo reprodutibilidade dos resultados
])

### Treino

In [ ]:
# Treinando o Pipeline da Árvore de Decisão com os dados de treino
dt_pipeline.fit(X_train, y_train) # O método fit ajusta/treina o modelo com os dados de treino, aplicando as transformações definidas no pipeline

### Gerando previsões

In [ ]:
# Gerando os palpites finais (0 ou 1) com a Árvore de Decisão
y_pred = dt_pipeline.predict(X_test) # Este método trás o resultado final

# Gerando as probabilidades de o vinho ser de alta qualidade (Classe 1)
y_proba = dt_pipeline.predict_proba(X_test)[:, 1] # Este método devolve o nível de certeza do modelo trazendo a segunda coluna (1)

### Avaliação das métricas

In [ ]:
# Calcula, exibe e guarda as métricas da Árvore de Decisão.
metricas_dt = avaliar_modelo(y_test, y_pred, y_proba)

### Matriz de confusão

In [ ]:
# Matriz de confusão e o relatório resumido
print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

### Plotando as informações do modelo

In [ ]:
# Configurando o tema visual e o tamanho do painel
sns.set_theme(style="whitegrid") # Configura o estilo visual dos gráficos
plt.rcParams['figure.figsize'] = (15, 6) # Define o tamanho padrão dos gráficos para 15 polegadas de largura e 6 polegadas de altura

# Criando o esqueleto com 3 sub-gráficos lado a lado
fig, (ax1, ax2, ax3) = plt.subplots(1, 3) # Cria uma figura com 3 subplots lado a lado

# Adicionando título principal do modelo
plt.suptitle("Modelo de Árvore de Decisão", fontsize=20, weight='bold', color='#2b0a0a') # Adiciona título principal do modelo com fonte maior, negrito e cor personalizada

# Plotando a Matriz de Confusão no primeiro eixo
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax1, cmap='Greens') # Plota a matriz de confusão usando as previsões do modelo e o conjunto de teste
ax1.set_title("Matriz de Confusão") # Titulo do gráfico da matriz de confusão

# Plotando a Curva ROC no segundo eixo
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax2, color='green') # Plota a curva ROC usando as probabilidades previstas pelo modelo e o conjunto de teste
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_title("Curva ROC (Sensibilidade)") # Titulo do gráfico da curva ROC

# Extraindo e plotando os coeficientes (impacto) no terceiro eixo
coefs = dt_pipeline.named_steps['model'].feature_importances_ # Acessa os coeficientes do modelo de árvore de decisão treinado, que indicam a importância de cada variável preditora na decisão do modelo
features = X_train.columns                        # Acessa os nomes das colunas do conjunto de treino, que correspondem às variáveis preditoras usadas no modelo
importancia = pd.Series(coefs, index=features).sort_values() # Cria uma série pandas com os coeficientes do modelo, indexada pelos nomes das variáveis preditoras ordenadas
importancia.plot(kind='barh', ax=ax3, color='mediumseagreen') # Plota um gráfico de barras horizontal mostrando a importância de cada característica na decisão do modelo
ax3.set_title("Impacto das Características") # Titulo do gráfico do impacto das características

# Ajustando o espaçamento entre os gráficos
plt.tight_layout()

# Salvando o gráfico automaticamente na pasta de resultados
plt.savefig(results_dir / 'arvore_decisao_outputs.png', dpi=300, bbox_inches='tight')

# Plotando o gráfico
plt.show()

### Análise de Desempenho: Árvore de Decisão

A Árvore de Decisão apresentou acurácia de **80%** no conjunto de teste.  
Esse modelo teve um comportamento interessante para a classe **Alta**, pois conseguiu identificar **26 dos 32 vinhos realmente classificados como alta qualidade**, alcançando um recall de **81%**.

Por outro lado, a precisão da classe Alta ficou em **39%**, indicando que o modelo também classificou muitos vinhos de baixa/média qualidade como se fossem de alta qualidade. A matriz de confusão mostra que houve **40 falsos positivos**, ou seja, vinhos que não eram de alta qualidade, mas foram classificados dessa forma pelo modelo.

Esse resultado mostra que a Árvore de Decisão foi mais sensível para encontrar vinhos de alta qualidade, mas menos seletiva na hora de confirmar essa classificação. Em comparação com a Regressão Logística, a Árvore de Decisão apresentou melhor recall e melhor F1-score para a classe Alta.

Apesar de não ser o modelo mais preciso, a Árvore de Decisão é útil para interpretação, pois permite entender melhor as regras e condições usadas para separar os vinhos entre baixa/média e alta qualidade.

## Modelo: Random Forest (Floresta Aleatória)

In [ ]:
# Montando a esteira Pipeline e ajustando os hiperparâmetros da floresta aleatória
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # Substitui valores ausentes pela mediana da coluna
    ('model', RandomForestClassifier(n_estimators=500, # Número de árvores na floresta
                                     class_weight='balanced', # Ajusta o peso das classes para lidar com desbalanceamento
                                     max_depth=None,  # Limita a profundidade máxima da árvore para evitar overfitting (memorização excessiva dos dados de treino)
                                     min_samples_split=4, # Número mínimo de amostras necessárias para dividir um nó interno
                                     min_samples_leaf=2, # Número mínimo de amostras necessárias para estar em um nó folha
                                     random_state=42)) # Semente aleatória para os cálculos se repetirem, garantindo reprodutibilidade dos resultados
])

### Treino

In [ ]:
# Treinando o Pipeline da Floresta aleatória com os dados de treino
rf_pipeline.fit(X_train, y_train) # O método fit ajusta/treina o modelo com os dados de treino, aplicando as transformações definidas no pipeline

### Previsões

In [ ]:
# Gerando os palpites finais (0 ou 1) com a Floresta aleatória
y_pred = rf_pipeline.predict(X_test) # Este método trás o resultado final

# Gerando as probabilidades de o vinho ser de alta qualidade (Classe 1)
y_proba = rf_pipeline.predict_proba(X_test)[:, 1] # Este método devolve o nível de certeza do modelo trazendo a segunda coluna (1)

### Avaliações métricas

In [ ]:
# Calcula, exibe e guarda as métricas do Random Forest.
metricas_rf = avaliar_modelo(y_test, y_pred, y_proba)

### Matriz de confusão

In [ ]:
# Matriz de confusão e o relatório resumido
print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

### Plotando os resultados 

In [ ]:
# Configurando o tema visual e o tamanho do painel
sns.set_theme(style="whitegrid") # Configura o estilo visual dos gráficos
plt.rcParams['figure.figsize'] = (15, 6) # Define o tamanho padrão dos gráficos para 15 polegadas de largura e 6 polegadas de altura

# Criando o esqueleto com 3 sub-gráficos lado a lado
fig, (ax1, ax2, ax3) = plt.subplots(1, 3) # Cria uma figura com 3 subplots lado a lado

# Adicionando título principal do modelo
plt.suptitle("Modelo de Floresta Aleatória (Random Forest)", fontsize=20, weight='bold', color='#1b4332') # Ajustado para o verde escuro

# Plotando a Matriz de Confusão no primeiro eixo
cmap_floresta = LinearSegmentedColormap.from_list("floresta", ["#f5f2eb", "#8b5a2b", "#1b4332"]) # Cria um mapa de cores personalizado para a matriz de confusão
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax1, cmap=cmap_floresta) # Plota a matriz de confusão usando as previsões do modelo e o conjunto de teste
ax1.set_title("Matriz de Confusão") # Titulo do gráfico da matriz de confusão

# Plotando a Curva ROC no segundo eixo
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax2, color='#1b4332') # Plota a curva ROC usando as probabilidades previstas pelo modelo e o conjunto de teste
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_title("Curva ROC (Sensibilidade)") # Titulo do gráfico da curva ROC

# Extraindo e plotando os coeficientes (impacto) no terceiro eixo
coefs = rf_pipeline.named_steps['model'].feature_importances_ # Acessa a importância de cada variável preditora na decisão do modelo
features = X_train.columns                        # Acessa os nomes das colunas do conjunto de treino
importancia = pd.Series(coefs, index=features).sort_values() # Cria uma série pandas com os coeficientes do modelo, indexada pelos nomes das variáveis preditoras ordenadas
importancia.plot(kind='barh', ax=ax3, color='#6f4e37') # Plota um gráfico de barras horizontal mostrando a importância (Marrom)
ax3.set_title("Impacto das Características") # Titulo do gráfico do impacto das características

# Ajustando o espaçamento entre os gráficos
plt.tight_layout()

# Salvando o gráfico automaticamente na pasta de resultados
plt.savefig(results_dir / 'floresta_aleatoria_outputs.png', dpi=300, bbox_inches='tight')

# Plotando o gráfico
plt.show()

### Análise de Desempenho: Random Forest

Após a etapa de *feature engineering*, o modelo **Random Forest** apresentou uma melhora importante no desempenho geral.  
Esse modelo utiliza várias árvores de decisão e combina seus resultados para gerar uma previsão final, o que tende a tornar a classificação mais estável do que uma única árvore.

### Principais resultados

- **Acurácia: 91,70%**  
  O modelo acertou a maior parte das classificações no conjunto de teste.

- **Precisão da classe Alta: 76,00%**  
  Quando o modelo classificou um vinho como de alta qualidade, ele acertou em 76% dos casos.

- **Recall da classe Alta: 59,38%**  
  Entre os vinhos que realmente eram de alta qualidade, o modelo conseguiu identificar aproximadamente 59%.

- **F1-score da classe Alta: 66,67%**  
  Essa métrica mostra um equilíbrio entre precisão e recall para a classe de alta qualidade.

- **ROC-AUC: 90,43%**  
  O modelo apresentou boa capacidade de separação entre vinhos de alta qualidade e vinhos de baixa/média qualidade.

### Interpretação

O Random Forest apresentou o melhor desempenho até o momento, principalmente em acurácia e ROC-AUC.  
Apesar disso, o recall da classe **Alta** ainda mostra que o modelo deixa de identificar parte dos vinhos de alta qualidade.

Esse comportamento pode estar relacionado ao desbalanceamento da base, já que a quantidade de vinhos classificados como alta qualidade é menor do que a quantidade de vinhos de baixa/média qualidade.

### Conclusão do módulo

Com os resultados obtidos, o Random Forest se mostrou um modelo forte para este problema de classificação.  
Ele conseguiu aproveitar melhor as variáveis originais e as novas variáveis criadas no processo de *feature engineering*, apresentando bom desempenho geral e boa capacidade de distinguir as classes.

Na próxima etapa, o modelo **SVM com kernel RBF** será avaliado para verificar se uma abordagem não linear consegue melhorar ainda mais a separação entre vinhos de alta qualidade e baixa/média qualidade.

## Modelo SVM (Support Vector Machines) com Kernel (Função de base radial)

In [ ]:
# Montando a esteira Pipeline e ajustando os hiperparâmetros do SVM
svm_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # Substitui valores ausentes pela mediana da coluna
    ('scaler', StandardScaler()), # Padroniza os dados para média 0 e desvio padrão 1 (campo obrigatório para o SVM)
    ('model', SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42))
])

### Treino

In [ ]:
# Treinando o Pipeline do SVM com os dados de treino
svm_pipeline.fit(X_train, y_train) # O método fit ajusta/treina o modelo com os dados de treino, aplicando as transformações definidas no pipeline

### Previsões

In [ ]:
# Gerando os palpites finais (0 ou 1) com a Árvore de Decisão
y_pred = svm_pipeline.predict(X_test) # Este método trás o resultado final

# Gerando as probabilidades de o vinho ser de alta qualidade (Classe 1)
y_proba = svm_pipeline.predict_proba(X_test)[:, 1] # Este método devolve o nível de certeza do modelo trazendo a segunda coluna (1)

### Avaliações métricas

In [ ]:
# Calcula, exibe e guarda as métricas do SVM.
metricas_svm = avaliar_modelo(y_test, y_pred, y_proba)

### Matriz de confusão

In [ ]:
# Matriz de confusão e o relatório resumido
print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

### Plotando os resultados

In [ ]:
# 1. Configurando o estilo visual e o tamanho do painel
sns.set_theme(style="whitegrid") # Configura o estilo visual dos gráficos
plt.rcParams['figure.figsize'] = (15, 6) # Define o tamanho padrão dos gráficos

# 2. Criando o esqueleto com 3 sub-gráficos lado a lado
fig, (ax1, ax2, ax3) = plt.subplots(1, 3) 

# Título principal em um tom de roxo profundo e elegante
plt.suptitle("Modelo de Máquinas de Vetores de Suporte (SVM RBF)", fontsize=16, weight='bold', color='#3b0764')

# 3. Gráfico 1: Matriz de Confusão (Customizada em degradê de Rosa para Roxo Escuro)
cmap_rosa_roxo = LinearSegmentedColormap.from_list("rosa_roxo", ["#fff0f5", "#ec4899", "#3b0764"])
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax1, cmap=cmap_rosa_roxo)
ax1.set_title("Matriz de Confusão")

# 4. Gráfico 2: Curva ROC (Linha em Roxo Vibrante)
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax2, color='#7209b7')
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_title("Curva ROC (Sensibilidade)")

# 5. Gráfico 3: Curva de Precisão-Recall (Linha em Rosa Choque/Magenta)
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=ax3, color='#f72585')
ax3.set_title("Curva de Precisão-Recall")

# 6. Ajustando o espaçamento entre os elementos
plt.tight_layout()

# 7. Salvando automaticamente o gráfico temático na sua pasta de resultados
plt.savefig(results_dir / 'svm_rbf_outputs.png', dpi=300, bbox_inches='tight')

# 8. Exibindo o painel na tela
plt.show()

### Análise de Desempenho: SVM (Support Vector Machines)

O modelo **SVM com kernel RBF** foi utilizado para testar uma abordagem mais flexível de separação entre as classes.  
Diferente de modelos mais simples, o SVM com RBF busca encontrar uma fronteira de decisão não linear, o que pode ajudar quando a diferença entre vinhos de alta qualidade e baixa/média qualidade depende da combinação de várias características.

O modelo apresentou acurácia de **82%** no conjunto de teste. Para a classe **Alta**, obteve **precision de 41%**, **recall de 69%** e **F1-score de 51%**.

A matriz de confusão mostra que o modelo acertou **22 dos 32 vinhos de alta qualidade**, mas também classificou **32 vinhos de baixa/média qualidade** como se fossem de alta qualidade. Isso indica que o modelo conseguiu encontrar uma parte relevante dos vinhos de alta qualidade, mas ainda apresentou dificuldade em evitar falsos positivos.

Em comparação com a Regressão Logística, o SVM apresentou melhor acurácia e melhor F1-score para a classe Alta. Porém, quando comparado ao Random Forest, seu desempenho foi inferior, principalmente em precisão e F1-score.

Dessa forma, o SVM com RBF mostrou um desempenho razoável, mas não superou o Random Forest como melhor modelo para este problema.

## 6. Conclusão final: avaliação comparativa e escolha do melhor modelo

Nesta etapa final, foram comparados os quatro modelos treinados para prever se um vinho pertence à classe de **alta qualidade** ou à classe de **baixa/média qualidade**, utilizando suas características físico-químicas.

Como a base possui desbalanceamento entre as classes, a avaliação não deve considerar apenas a acurácia geral. A classe de vinhos de alta qualidade possui menos registros, por isso também foram analisadas métricas como **precision**, **recall**, **F1-score** e **ROC-AUC**.

---

### Quadro comparativo geral dos modelos

A tabela abaixo resume o desempenho obtido por cada modelo no conjunto de teste:

| Modelo de Classificação | Acurácia Geral | Precision Classe Alta | Recall Classe Alta | F1-Score Classe Alta | ROC-AUC | Observação |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **Regressão Logística** | 78,60% | 35,59% | 65,62% | 46,15% | 87,01% | Modelo base, com boa capacidade de separação, mas baixa precisão para a classe Alta. |
| **Árvore de Decisão** | 79,91% | 39,39% | **81,25%** | 53,06% | 82,35% | Modelo com maior recall, ou seja, encontrou mais vinhos de alta qualidade. |
| **Random Forest** | **91,70%** | **76,00%** | 59,38% | **66,67%** | **90,43%** | Melhor desempenho geral e melhor equilíbrio entre as métricas. |
| **SVM com RBF** | 81,66% | 40,74% | 68,75% | 51,16% | 86,62% | Resultado intermediário, melhor que a Regressão Logística em F1-score. |

---

### Análise crítica dos modelos

#### 1. Regressão Logística

A Regressão Logística foi utilizada como modelo inicial de referência, por ser mais simples e interpretável.

O modelo apresentou **78,60% de acurácia**. Para a classe **Alta**, obteve **precision de 35,59%**, **recall de 65,62%**, **F1-score de 46,15%** e **ROC-AUC de 87,01%**.

A matriz de confusão mostrou que o modelo conseguiu identificar parte dos vinhos de alta qualidade, mas também classificou muitos vinhos de baixa/média qualidade como se fossem de alta qualidade.

Esse comportamento indica que a Regressão Logística foi útil como modelo base, mas teve dificuldade em ser precisa ao indicar vinhos da classe Alta.



#### 2. Árvore de Decisão

A Árvore de Decisão apresentou **79,91% de acurácia**. Para a classe **Alta**, obteve **precision de 39,39%**, **recall de 81,25%**, **F1-score de 53,06%** e **ROC-AUC de 82,35%**.

Esse foi o modelo com o maior **recall** para a classe Alta, conseguindo identificar a maior parte dos vinhos realmente classificados como de alta qualidade.

Por outro lado, a precision ficou baixa, indicando que o modelo também classificou muitos vinhos de baixa/média qualidade como se fossem de alta qualidade.

Dessa forma, a Árvore de Decisão foi mais sensível para encontrar vinhos de alta qualidade, mas menos seletiva na hora de confirmar essa classificação.



#### 3. Random Forest

O modelo **Random Forest** apresentou o melhor desempenho geral entre os modelos avaliados.

Após o processo de *feature engineering*, o modelo atingiu **91,70% de acurácia**, **76,00% de precision para a classe Alta**, **59,38% de recall**, **66,67% de F1-score** e **90,43% de ROC-AUC**.

Apesar de não ter apresentado o maior recall, o Random Forest foi o modelo mais equilibrado. Ele teve a maior precisão para a classe Alta, o maior F1-score e a melhor acurácia geral.

Isso mostra que o modelo foi mais seletivo: ele deixou de identificar alguns vinhos de alta qualidade, mas quando classificou um vinho como Alta, teve uma taxa de acerto bem superior aos demais modelos.



#### 4. SVM com RBF

O SVM com kernel RBF apresentou **81,66% de acurácia**. Para a classe **Alta**, obteve **precision de 40,74%**, **recall de 68,75%**, **F1-score de 51,16%** e **ROC-AUC de 86,62%**.

O resultado foi intermediário. O SVM superou a Regressão Logística em acurácia, precision e F1-score da classe Alta, mas ficou abaixo do Random Forest.

Esse modelo mostrou capacidade razoável de separar as classes, mas ainda apresentou dificuldade em evitar falsos positivos.

---

### Modelo recomendado: Random Forest

Com base nos resultados obtidos, o modelo recomendado é o **Random Forest**.

A escolha se justifica por três pontos principais:

1. **Maior acurácia geral**  
   O modelo atingiu **91,70% de acerto** no conjunto de teste, superando os demais modelos avaliados.

2. **Maior precisão para a classe Alta**  
   A precision da classe Alta foi de **76,00%**, indicando que, quando o modelo classificou um vinho como alta qualidade, ele acertou com mais frequência do que os demais modelos.

3. **Melhor equilíbrio entre as métricas**  
   O Random Forest apresentou o melhor **F1-score da classe Alta**, com **66,67%**, além do maior **ROC-AUC**, com **90,43%**. Isso indica boa capacidade de separação entre as classes e melhor equilíbrio entre precision e recall.

---

### Interpretação dos resultados

Os modelos apresentaram comportamentos diferentes.

A **Árvore de Decisão** foi o modelo que mais encontrou vinhos de alta qualidade, pois obteve o maior recall da classe Alta. Porém, também gerou muitos falsos positivos, o que reduziu sua precision.

A **Regressão Logística** funcionou bem como modelo base, mas teve o menor desempenho em precision e F1-score para a classe Alta.

O **SVM com RBF** apresentou desempenho intermediário, com melhor resultado que a Regressão Logística, mas ainda abaixo do Random Forest.

O **Random Forest** foi o modelo mais equilibrado. Ele não teve o maior recall, mas apresentou a melhor precision, o melhor F1-score, a maior acurácia geral e o maior ROC-AUC.

---

### Interpretação das características no processo

Com base no modelo Random Forest, também é possível analisar quais variáveis tiveram maior importância para a classificação.

As variáveis físico-químicas originais e as novas variáveis criadas no processo de *feature engineering* ajudaram o modelo a capturar relações mais completas entre os atributos dos vinhos.

Entre as variáveis que podem ter maior relevância para a classificação, estão:

- `alcohol`;
- `sulphates`;
- `volatile acidity`;
- `density`;
- `alcohol_density_ratio`;
- `sulphates_chlorides_ratio`;
- `alcohol_sulphates`.

Essas variáveis podem indicar padrões associados à qualidade do vinho. Porém, é importante destacar que o modelo identifica **associações estatísticas**, e não relações de causa e efeito.

Ou seja, o modelo aponta quais características ajudam na previsão, mas não prova que uma variável isolada causa diretamente o aumento da qualidade do vinho.

---

### Próximos passos recomendados

Como próximos passos, seria interessante:

1. **Ajustar hiperparâmetros do Random Forest**  
   Testar combinações com `GridSearchCV` ou `RandomizedSearchCV`, avaliando parâmetros como `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf` e `max_features`.

2. **Avaliar diferentes limites de classificação**  
   O modelo usa por padrão o limite de 0.5 para classificar um vinho como alta qualidade. Alterar esse limite pode melhorar o equilíbrio entre precision e recall.

3. **Aplicar validação cruzada**  
   A validação cruzada ajudaria a verificar se o desempenho do modelo se mantém estável em diferentes divisões da base.

4. **Incluir novas variáveis externas**  
   Caso estivessem disponíveis, informações como safra, região, tempo de maturação, tipo de uva e avaliações sensoriais poderiam enriquecer o modelo.

---

### Conclusão final

O projeto demonstrou que é possível utilizar características físico-químicas dos vinhos para construir modelos capazes de prever a qualidade de forma automatizada.

Entre os modelos testados, o **Random Forest** apresentou o melhor desempenho geral após o processo de *feature engineering*. Ele foi o modelo mais equilibrado, com maior acurácia, maior precision para a classe Alta, maior F1-score e maior ROC-AUC.

Apesar disso, o recall da classe Alta ainda pode ser melhorado, já que o modelo deixou de identificar parte dos vinhos realmente classificados como alta qualidade.

Dessa forma, o Random Forest é o melhor candidato entre os modelos avaliados, mas ainda pode ser aprimorado com ajuste de hiperparâmetros, validação cruzada e avaliação de diferentes limites de classificação.